Dataset só consegue responder, qual o item na posição [X]. Alguem precisa decidir quais posições pedir e qual a ordem, esse é o papel do sampler

In [ ]:
import torch    
from torch.utils.data import TensorDataset


x, y = torch.rand(10, 2), torch.rand(10, 1)
dataset = TensorDataset(x, y)

# 1. SequentialSampler

Don't randomizer the idx of the dataset

In [7]:
from torch.utils.data import SequentialSampler

sampler = SequentialSampler(dataset)

print(list(sampler))
print(len(sampler))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
10


# 2. RandomSampler

Randomizer the idx of the dataset

In [6]:
from torch.utils.data import RandomSampler

sampler = RandomSampler(dataset)

print(list(sampler))
print(len(sampler))

[2, 3, 0, 8, 5, 7, 9, 4, 1, 6]
10


# 3. DistributedSampler 

Used in multi-GPU training (`DistributedDataParallel`). Ensures each GPU processes a different slice of the dataset, with no overlap between them.

```python
sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
loader = DataLoader(dataset, batch_size=32, sampler=sampler)
```

- `num_replicas` → total number of processes/GPUs.
- `rank` → the id of this specific process (0, 1, 2...).

## How it works internally

**1. Every rank computes the same full shuffled list.**

This happens because all ranks use the same seed (default `seed=0` + `epoch`). Same seed → same `torch.randperm` → identical result on every process, with no communication needed between them.

```python
g = torch.Generator()
g.manual_seed(seed + epoch)
indices = torch.randperm(len(dataset), generator=g).tolist()
# this list is IDENTICAL across all ranks
```

**2. Each rank slices out its own portion** of that same list, using strided slicing:

```python
indices[rank::num_replicas]
```

Example with 10 items and 4 ranks, shuffled list `[7,2,9,0,5,3,8,1,6,4]`:

```
rank 0 → positions 0,4,8 → [7, 5, 6]
rank 1 → positions 1,5,9 → [2, 3, 4]
rank 2 → positions 2,6   → [9, 8]
rank 3 → positions 3,7   → [0, 1]
```

Combined, all slices cover the entire dataset, with no overlap.

## Why there's no overlap between ranks

Because every rank starts from the **same base list** (same seed) and each one takes a different slice by position (`rank::num_replicas`). If each rank generated its own independent random sequence, indices could accidentally repeat across ranks — starting from the same base list avoids that.

## Padding (when it doesn't divide evenly)

If the dataset doesn't divide evenly across ranks (e.g. 10 items, 3 ranks), the sampler repeats a few indices from the start of the list to make sure every rank ends up with the **same number of items** — needed to avoid desynchronization between GPUs (one finishing before the others).

## set_epoch — don't forget it

The effective seed used is `seed + epoch`. Without calling this, `epoch` stays `0`, and the shuffle never changes between epochs (a common, silent bug):

```python
for epoch in range(num_epochs):
    sampler.set_epoch(epoch)   # changes the shuffle each epoch
    for x_batch, y_batch in loader:
        ...
```

## Summary of the mechanism

**Determinism (same seed → same full list) + strided slicing by rank (`[rank::num_replicas]`) = each GPU processes a different slice, with no overlap, with no need for the processes to communicate with each other.**

In [34]:
from torch.utils.data import DistributedSampler

sampler = DistributedSampler(
    dataset = dataset,
    num_replicas = 2, # Divide the total by 2, and only get the result of the division of the ranls
    shuffle = False,
    rank = 0
)

print(list(sampler))

[0, 2, 4, 6, 8]


In [29]:
num_replicas = 2

for rank in range(num_replicas):
    sampler = DistributedSampler(dataset, num_replicas=num_replicas, rank=rank, shuffle=False, seed = 42)
    print(f"rank {rank}: {list(sampler)}")

rank 0: [0, 2, 4, 6, 8]
rank 1: [1, 3, 5, 7, 9]
